In [ ]:
import torch
import numpy as np
import pandas as pd
from scipy.spatial.distance import pdist, squareform
from scipy.optimize import curve_fit
from scipy.stats import zscore

In [ ]:
# Step 0: 从 Excel 文件中读取脑区坐标和基因表达矩阵
coordinates_df = pd.read_csv(r"E:\aliyun_backup\muilt_disorders\01_altas\BNA246_Cerebellum_Stem_Asym.csv")
coordinates = coordinates_df[['MNI_X', 'MNI_Y', 'MNI_Z']].values  # 转换为 numpy 数组

gene_expression_df = pd.read_csv(r"E:\aliyun_backup\muilt_disorders\13_Allen\AHBA_expression_BNA246.csv")  # 假设基因表达矩阵文件
gene_expression_df = gene_expression_df.drop(columns='label')
gene_names = gene_expression_df.columns  # 基因名作为列名

gene_expression_matrix = gene_expression_df.values  # 转换为 numpy 数组

In [ ]:
# Step 1: 计算每对脑区的欧氏距离矩阵
distance_matrix = squareform(pdist(coordinates, metric='euclidean'))

# Step 2: 对基因表达矩阵按行进行 z-score 标准化
z_gene_expression = np.apply_along_axis(zscore, 1, gene_expression_matrix)


In [ ]:
# Step 3: 将距离矩阵和基因表达矩阵转换为 PyTorch 张量，并移动到 GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
distance_matrix = torch.tensor(distance_matrix, dtype=torch.float32, device=device)
z_gene_expression = torch.tensor(z_gene_expression, dtype=torch.float32, device=device)



In [ ]:
distance_matrix

In [ ]:
# Step 4: 定义指数衰减函数，形式为 A * e^(-d / n) + B
def exponential_decay(d, A, n, B):
    return A * np.exp(-d / n) + B

In [ ]:
# Step 5: 计算每对脑区之间的基因表达相关性矩阵 (CGE)
correlation_matrix = torch.corrcoef(z_gene_expression)

In [ ]:
# Step 6: 提取距离矩阵和相关性矩阵的上三角元素
upper_tri_indices = torch.triu_indices(distance_matrix.size(0), distance_matrix.size(1), offset=1)
distances = distance_matrix[upper_tri_indices[0], upper_tri_indices[1]]
correlations = correlation_matrix[upper_tri_indices[0], upper_tri_indices[1]]


In [ ]:
# Step 7: 使用 SciPy 的 curve_fit 进行非线性拟合，以获得参数 A, n, 和 B
# 注意: curve_fit 不支持 GPU 张量，因此这里需转换回 CPU
distances_cpu = distances.cpu().numpy()
correlations_cpu = correlations.cpu().numpy()
# 设置初始参数
initial_params = [0.64, 90.4, -0.19]
# 执行拟合
popt, _ = curve_fit(exponential_decay, distances_cpu, correlations_cpu, p0=initial_params)
A, n, B = popt
print("拟合参数 A:", A)
print("拟合参数 n:", n)
print("拟合参数 B:", B)

In [ ]:
import matplotlib.pyplot as plt

# 可视化拟合效果
plt.figure(figsize=(8, 6))
plt.scatter(distances_cpu, correlations_cpu, s=1, color='gray', alpha=0.5, label='Data')
sorted_indices = np.argsort(distances_cpu)
plt.plot(distances_cpu[sorted_indices], exponential_decay(distances_cpu[sorted_indices], *popt), 
         color='red', linewidth=2, label='Fitted Curve')
plt.xlabel("Distance between regions (mm)")
plt.ylabel("Correlated gene expression (CGE)")
plt.title("CGE vs. Distance with Fitted Exponential Decay")
plt.legend()
# plt.savefig(r'E:\aliyun_backup\common_individual_166\11_ALLEN_avg\CGE_vs_Distance.tif',dpi=300)
plt.show()

In [ ]:
# Step 8: 使用 PyTorch 重新创建拟合参数并在 GPU 上计算期望值矩阵
A = torch.tensor(A, device=device)
n = torch.tensor(n, device=device)
B = torch.tensor(B, device=device)

# 初始化 expected_matrix 张量并计算距离相关的期望值矩阵
expected_matrix = torch.zeros_like(distance_matrix)
for i in range(distance_matrix.size(0)):
    for j in range(i + 1, distance_matrix.size(1)):
        expected_matrix[i, j] = A * torch.exp(-distance_matrix[i, j] / n) + B
        expected_matrix[j, i] = expected_matrix[i, j]  # 对称矩阵

In [ ]:
pd.DataFrame(expected_matrix.cpu().numpy())

In [ ]:
# Step 9: 构建存储每条连接基因贡献度的 DataFrame
num_regions = z_gene_expression.size(0)  # 160个脑区
num_genes = z_gene_expression.size(1)    # 10000个基因
num_connections = distances.size(0)      # 上三角元素数量

# 创建 gene_contribution_df 数据框架
gene_contribution_df = pd.DataFrame(np.zeros((num_connections, num_genes)), 
                                    columns=gene_names)  # 使用基因名作为列名


In [ ]:
# Step 10: 计算每条连接的基因贡献度
region_pairs = list(zip(upper_tri_indices[0].cpu().numpy(), upper_tri_indices[1].cpu().numpy()))

# 对每个连接计算基因贡献度
for idx, (i, j) in enumerate(region_pairs):
    product_z_scores = z_gene_expression[i] * z_gene_expression[j]
    contribution_scores = product_z_scores - expected_matrix[i, j]
    gene_contribution_df.iloc[idx, :] = contribution_scores.cpu().numpy()  # 从 GPU 移动到 CPU


In [ ]:
# 为 gene_contribution_df 添加连接信息列
gene_contribution_df.insert(0, 'Region_Pair', [f'{i}-{j}' for i, j in region_pairs])
gene_contribution_df

In [ ]:
# gene_contribution_df.to_csv(r'E:\aliyun_backup\muilt_disorders\13_Allen\gene_contribution_df.csv', index=False)